# Cross-Phase Summary

- One row per (phase, gas): bins, BN MAE, skill_vs_baseline (vs. hour-only), gap_closed

- Phase-to-phase columns: skill_vs_baseline(mae_3a, mae_2) (what cross-sensitivity adds) and skill_vs_baseline(mae_3b, mae_3a) (what the latent node adds).

- Only compare gap_closed within a bin count because the floor changes with the number of bins.

In [ ]:
import pandas as pd

from cse250a.modules.config import config
from cse250a.modules.eval.skill import skill_table, skill_vs_baseline

PHASES = {"Phase 1": "phase1", "Phase 2": "phase2", "Phase 3a": "phase3a", "Phase 3b": "phase3b"}
cols = ["bins", "seed", "gas", "model", "MAE", "RMSE"]

grids = {}
for label, folder in PHASES.items():
    g = pd.read_csv(config.paths.result(folder, "concentration_grid.csv"))[cols]
    grids[label] = g[g["model"].isin(["BN", "HourOnlyBaseline", "resolution_floor"])]

In [ ]:
for label, g in grids.items():
    print(label, sorted(g["bins"].unique()), g["seed"].nunique(), "seeds")

In [ ]:
def ms(s):
    return f"{s.mean():.3f} ± {s.std():.3f}"

rows = []
for label, g in grids.items():
    sk = skill_table(g)                    # per (bins, seed, gas)
    for (bins, gas), d in sk.groupby(["bins", "gas"]):
        rows.append({
            "phase": label, "gas": gas, "bins": bins,
            "BN MAE": ms(d["BN"]),
            "skill_vs_baseline": ms(d["skill_vs_baseline"]),
            "gap_closed": ms(d["gap_closed"]),
        })
summary = pd.DataFrame(rows)


In [ ]:
def mae_of(label):
    g = grids[label]
    return (g[g["model"] == "BN"]
            .set_index(["bins", "seed", "gas"])["MAE"].rename(label))

mae = pd.concat([mae_of(p) for p in PHASES], axis=1)

def step(new, old):
    d = mae.dropna(subset=[new, old])
    s = d.apply(lambda r: skill_vs_baseline(r[new], r[old]), axis=1)
    return s.groupby(["bins", "gas"]).agg(["mean", "std"]).apply(
        lambda r: f"{r['mean']:.3f} ± {r['std']:.3f}", axis=1)

adds = pd.DataFrame({
    "3a vs 2 (what cross edges add)": step("Phase 3a", "Phase 2"),
    "3b vs 3a (what latent node adds)": step("Phase 3b", "Phase 3a"),
}).reset_index()

summary = (summary
    .merge(adds.assign(phase="Phase 3a")[["bins", "gas", "phase", "3a vs 2 (what cross edges add)"]],
           on=["bins", "gas", "phase"], how="left")
    .merge(adds.assign(phase="Phase 3b")[["bins", "gas", "phase", "3b vs 3a (what latent node adds)"]],
           on=["bins", "gas", "phase"], how="left"))
out = summary.sort_values(["bins", "gas", "phase"]).fillna("")
out

In [ ]:
out.to_csv(config.paths.result("summary", "cross_phase_summary.csv"), index=False)

### Interpretation

- The difference between phase 1 and phase 3b's skill_vs_baseline and gap_closed is minimal for CO, the only gas that the two phases share. 